In [1]:
import numpy as np
import pandas as pd
import torch
from torch import nn
from torch.utils.data import DataLoader, random_split
from sklearn.preprocessing import OneHotEncoder, StandardScaler
from sklearn.impute import SimpleImputer



class MYCSVDataSet(torch.utils.data.Dataset):
    def __init__(self, csv_path, target_col):
        super().__init__()
        self.CSVDataSet = pd.read_csv(csv_path)

        features = self.CSVDataSet.drop(target_col, axis=1)
        self.y = self.CSVDataSet[target_col]

        categorical_columns = features.select_dtypes(include=["object", "category"]).columns.tolist()
        numerical_columns = features.select_dtypes(include=["int64", "float64"]).columns.tolist()


        self.CSVScaler = StandardScaler()
        if numerical_columns:
            self.NumImputer = SimpleImputer(strategy="median")
            numerical_filled = self.NumImputer.fit_transform(features[numerical_columns])
            scaled_numerical_data = self.CSVScaler.fit_transform(numerical_filled)
        else:
            scaled_numerical_data = np.empty((len(features), 0))


        self.CSVEncoder = OneHotEncoder(sparse_output=False)
        if categorical_columns:
            self.CatImputer = SimpleImputer(strategy="most_frequent")
            categorical_filled = self.CatImputer.fit_transform(features[categorical_columns])
            encoded_categorical_data = self.CSVEncoder.fit_transform(categorical_filled)
        else:
            encoded_categorical_data = np.empty((len(features), 0))

        self.x = np.hstack((scaled_numerical_data, encoded_categorical_data))
        self.x = torch.from_numpy(self.x.astype(np.float32))
        
        if str(self.y.dtype).startswith("int") or self.y.nunique() < 20:
            self.y = torch.from_numpy(self.y.values.astype(np.int64))
        else:
            self.y = torch.from_numpy(self.y.values.astype(np.float32))        

    def __len__(self):
        return self.x.shape[0]

    def __getitem__(self, index):
        x = self.x[index]
        y = self.y[index]
        return x, y


class Trainer:
    def __init__(self, model, optimizer, loss_fn, device):
        self.model = model.to(device)
        self.optimizer = optimizer
        self.loss_fn = loss_fn
        self.device = device

    def fit(self, train_loader, val_loader, epochs):
        best_val_loss = float("inf")
        for epoch in range(epochs):
            train_loss = self.train_one_epoch(train_loader)
            val_loss = self.evaluate(val_loader)

            print(f"Epoch {epoch+1}/{epochs} | train_loss: {train_loss:.4f} | val_loss: {val_loss:.4f}")

            if val_loss < best_val_loss:
                best_val_loss = val_loss
                self.save_checkpoint("best_model.pt")

    def train_one_epoch(self, loader):
        self.model.train()
        total_loss = 0.0

        for x, y in loader:
            x, y = x.to(self.device), y.to(self.device)

            self.optimizer.zero_grad()
            preds = self.model(x)
            loss = self.loss_fn(preds, y)
            loss.backward()
            self.optimizer.step()

            total_loss += loss.item() * x.size(0)

        return total_loss / len(loader.dataset)

    def evaluate(self, loader):
        self.model.eval()
        total_loss = 0.0

        with torch.no_grad():
            for x, y in loader:
                x, y = x.to(self.device), y.to(self.device)
                preds = self.model(x)
                loss = self.loss_fn(preds, y)
                total_loss += loss.item() * x.size(0)

        return total_loss / len(loader.dataset)


    def save_checkpoint(self, path):
        torch.save({
            "model_state": self.model.state_dict(),
            "optimizer_state": self.optimizer.state_dict(),
        }, path)

    def load_checkpoint(self, path):
        checkpoint = torch.load(path)
        self.model.load_state_dict(checkpoint["model_state"])
        self.optimizer.load_state_dict(checkpoint["optimizer_state"])




Check the column names

In [2]:
data_frame = pd.read_csv("fashion-mnist_train.csv")
print(data_frame.columns.tolist())
print(data_frame.head())

['label', 'pixel1', 'pixel2', 'pixel3', 'pixel4', 'pixel5', 'pixel6', 'pixel7', 'pixel8', 'pixel9', 'pixel10', 'pixel11', 'pixel12', 'pixel13', 'pixel14', 'pixel15', 'pixel16', 'pixel17', 'pixel18', 'pixel19', 'pixel20', 'pixel21', 'pixel22', 'pixel23', 'pixel24', 'pixel25', 'pixel26', 'pixel27', 'pixel28', 'pixel29', 'pixel30', 'pixel31', 'pixel32', 'pixel33', 'pixel34', 'pixel35', 'pixel36', 'pixel37', 'pixel38', 'pixel39', 'pixel40', 'pixel41', 'pixel42', 'pixel43', 'pixel44', 'pixel45', 'pixel46', 'pixel47', 'pixel48', 'pixel49', 'pixel50', 'pixel51', 'pixel52', 'pixel53', 'pixel54', 'pixel55', 'pixel56', 'pixel57', 'pixel58', 'pixel59', 'pixel60', 'pixel61', 'pixel62', 'pixel63', 'pixel64', 'pixel65', 'pixel66', 'pixel67', 'pixel68', 'pixel69', 'pixel70', 'pixel71', 'pixel72', 'pixel73', 'pixel74', 'pixel75', 'pixel76', 'pixel77', 'pixel78', 'pixel79', 'pixel80', 'pixel81', 'pixel82', 'pixel83', 'pixel84', 'pixel85', 'pixel86', 'pixel87', 'pixel88', 'pixel89', 'pixel90', 'pixel91'

Create the dataset and loader

In [3]:
dataset = MYCSVDataSet("fashion-mnist_train.csv", target_col="label")
loader = DataLoader(dataset, batch_size=32, shuffle=True)


Split the dataset as train and test.

In [4]:
train_size = int(0.8 * len(dataset))
val_size = len(dataset) - train_size
train_dataset, val_dataset = random_split(dataset, [train_size, val_size])


Preparing the model

In [13]:
train_loader = DataLoader(train_dataset, batch_size=32, shuffle=True)
val_loader = DataLoader(val_dataset, batch_size=32, shuffle=False)


model = nn.Sequential(
    nn.Linear(dataset.x.shape[1], 32),
    nn.ReLU(),
    nn.Linear(32, 10)
)
optimizer = torch.optim.Adam(model.parameters(), lr=0.0001)
loss_fn = nn.CrossEntropyLoss() #Use a different loss function for a different dataset.

trainer = Trainer(model, optimizer, loss_fn, device="cpu")

Start

In [ ]:
trainer.fit(train_loader, val_loader, epochs=10) #batch size 32 lr=0.001

Epoch 1/10 | train_loss: 0.4802 | val_loss: 0.3813
Epoch 2/10 | train_loss: 0.3669 | val_loss: 0.3568
Epoch 3/10 | train_loss: 0.3324 | val_loss: 0.3493
Epoch 4/10 | train_loss: 0.3106 | val_loss: 0.3604
Epoch 5/10 | train_loss: 0.2917 | val_loss: 0.3573
Epoch 6/10 | train_loss: 0.2761 | val_loss: 0.3668
Epoch 7/10 | train_loss: 0.2678 | val_loss: 0.3470
Epoch 8/10 | train_loss: 0.2547 | val_loss: 0.3615
Epoch 9/10 | train_loss: 0.2446 | val_loss: 0.3589
Epoch 10/10 | train_loss: 0.2385 | val_loss: 0.3718


In [8]:
trainer.fit(train_loader, val_loader, epochs=10) #batch size 64 lr=0.001

Epoch 1/10 | train_loss: 0.5105 | val_loss: 0.4018
Epoch 2/10 | train_loss: 0.3754 | val_loss: 0.3782
Epoch 3/10 | train_loss: 0.3413 | val_loss: 0.3799
Epoch 4/10 | train_loss: 0.3189 | val_loss: 0.3578
Epoch 5/10 | train_loss: 0.3010 | val_loss: 0.3480
Epoch 6/10 | train_loss: 0.2890 | val_loss: 0.3523
Epoch 7/10 | train_loss: 0.2770 | val_loss: 0.3641
Epoch 8/10 | train_loss: 0.2671 | val_loss: 0.3419
Epoch 9/10 | train_loss: 0.2565 | val_loss: 0.3359
Epoch 10/10 | train_loss: 0.2479 | val_loss: 0.3575


In [10]:
trainer.fit(train_loader, val_loader, epochs=10) #batch size 64 lr=0.01

Epoch 1/10 | train_loss: 0.5374 | val_loss: 0.4956
Epoch 2/10 | train_loss: 0.4599 | val_loss: 0.5340
Epoch 3/10 | train_loss: 0.4392 | val_loss: 0.4579
Epoch 4/10 | train_loss: 0.4280 | val_loss: 0.4234
Epoch 5/10 | train_loss: 0.4108 | val_loss: 0.5128
Epoch 6/10 | train_loss: 0.4044 | val_loss: 0.5200
Epoch 7/10 | train_loss: 0.4038 | val_loss: 0.5203
Epoch 8/10 | train_loss: 0.3869 | val_loss: 0.5571
Epoch 9/10 | train_loss: 0.3958 | val_loss: 0.4862
Epoch 10/10 | train_loss: 0.3749 | val_loss: 0.5410


In [12]:
trainer.fit(train_loader, val_loader, epochs=10) #batch size 16 lr=0.001

Epoch 1/10 | train_loss: 0.4620 | val_loss: 0.3904
Epoch 2/10 | train_loss: 0.3658 | val_loss: 0.3734
Epoch 3/10 | train_loss: 0.3318 | val_loss: 0.3535
Epoch 4/10 | train_loss: 0.3095 | val_loss: 0.3619
Epoch 5/10 | train_loss: 0.2951 | val_loss: 0.3598
Epoch 6/10 | train_loss: 0.2805 | val_loss: 0.3605
Epoch 7/10 | train_loss: 0.2702 | val_loss: 0.3740
Epoch 8/10 | train_loss: 0.2599 | val_loss: 0.3921
Epoch 9/10 | train_loss: 0.2524 | val_loss: 0.3745
Epoch 10/10 | train_loss: 0.2445 | val_loss: 0.3838


In [14]:
trainer.fit(train_loader, val_loader, epochs=10) #batch size 32 lr=0.0001

Epoch 1/10 | train_loss: 0.8027 | val_loss: 0.5253
Epoch 2/10 | train_loss: 0.4809 | val_loss: 0.4521
Epoch 3/10 | train_loss: 0.4293 | val_loss: 0.4206
Epoch 4/10 | train_loss: 0.4018 | val_loss: 0.4063
Epoch 5/10 | train_loss: 0.3834 | val_loss: 0.3943
Epoch 6/10 | train_loss: 0.3701 | val_loss: 0.3873
Epoch 7/10 | train_loss: 0.3589 | val_loss: 0.3791
Epoch 8/10 | train_loss: 0.3498 | val_loss: 0.3731
Epoch 9/10 | train_loss: 0.3422 | val_loss: 0.3691
Epoch 10/10 | train_loss: 0.3348 | val_loss: 0.3631
